# Intermediate Concepts 

Building on core concepts, explore advanced features of LangGraph.

## Advanced State Management

In basic examples, we used normal dictionaries for state.

But in larger projects,
it becomes difficult to manage state properly.

So we use TypedDict.

TypedDict helps us define:
- what data exists in state
- expected data types
- better structure and readability

Example:
- messages → list
- memory → dictionary
- counter → integer

This makes state management cleaner and easier to maintain.

In [2]:
from typing import TypedDict

class AgentState(TypedDict):
    messages: list[str]
    memory: dict
    counter: int

# Example
state: AgentState = {"messages": [], "memory": {}, "counter": 0}
print(state)

{'messages': [], 'memory': {}, 'counter': 0}


## Multi-actor Graphs
Graphs with multiple agents interacting.


A multi-actor graph contains multiple agents or nodes working together.

Instead of one AI handling everything,
different agents can perform different tasks.

Example:
- One agent does research
- One agent summarizes results
- One agent checks quality

This makes workflows more modular and scalable.

Flow:

Agent1 → Agent2 → END

- Agent1 performs first task
- Agent2 performs next task
- Final result is returned

In [3]:
from langgraph.graph import StateGraph, END

# Nodes
def agent1(state):
    state["messages"].append("Agent1 response")     # add agent1 response to messages list

    # Save to memory
    state["memory"]["agent1"] = "completed"        # save agent 1 status to memory

    return state                                   # return the updated state

def agent2(state):
    state["messages"].append("Agent2 response")

    # Save to memory
    state["memory"]["agent2"] = "completed"

    return state


# Graph
graph = StateGraph(dict)  # create a state graph with dict as shared state

graph.add_node("agent1", agent1)
graph.add_node("agent2", agent2)

graph.set_entry_point("agent1")

graph.add_edge("agent1", "agent2")
graph.add_edge("agent2", END)

# Compile
app = graph.compile()

# Run
result = app.invoke({
    "messages": [],
    "memory": {},
    "counter": 0
})

print(result) 

{'messages': ['Agent1 response', 'Agent2 response'], 'memory': {'agent1': 'completed', 'agent2': 'completed'}, 'counter': 0}


## Error Handling

In real applications,
errors can happen anytime.

Examples:
- API failure
- Invalid input
- Tool errors

So we use try-except blocks to safely handle errors.

Instead of crashing the workflow,
we can:
- catch errors
- save error messages
- retry steps
- continue execution

In [4]:
def risky_node(state):
    try:
        # Simulate error
        if state["counter"] > 5:
            raise ValueError("Counter too high")

        state["counter"] += 1

        # Success memory
        state["memory"]["status"] = "success"

        return state

    except Exception as e:
        # Save error message
        state["messages"].append(f"Error: {e}")

        # Store error in memory
        state["memory"]["last_error"] = str(e)

        return state
# Test
state = {
    "messages": [],
    "memory": {},
    "counter": 6
}

result = risky_node(state)

print(result)

{'messages': ['Error: Counter too high'], 'memory': {'last_error': 'Counter too high'}, 'counter': 6}


## Persistence

Persistence means saving state permanently.

Normally, state exists only while the program is running.

But with persistence,
we can save state into files or databases.

This allows:
- resuming conversations later
- restoring memory
- continuing workflows after restart

Save State → File → Load State Later

“In this example, we save state using JSON.”

In [6]:
import json

def save_state(state, filename="state.json"):
    with open(filename, "w") as f:
        json.dump(state, f)

def load_state(filename="state.json"):
    try:
        with open(filename, "r") as f:
            return json.load(f)
    except FileNotFoundError:
        return {"messages": [], "memory": {}, "counter": 0}


# Example
state = {"messages": ["Hello"], "memory": {"key": "value"}, "counter": 1}
save_state(state)
loaded = load_state()
print(loaded)        

{'messages': ['Hello'], 'memory': {'key': 'value'}, 'counter': 1}


## Human-in-the-Loop

Sometimes AI should not make decisions completely on its own.

A human may need to:
- review output
- approve actions
- provide extra input

LangGraph allows humans to participate inside the workflow.

This is called Human-in-the-Loop.

```
AI generates response
↓
Human reviews it
↓
Workflow continues

```

In [ ]:
def human_input_node(state):
    user_input = input("Human input: ")
    state["messages"].append(user_input)
    return state

# Add to graph
graph.add_node("human", human_input_node)
# Connect as needed
# example 
# graph.add_edge("llm", "human")
# graph.add_edge("human", "next_node")

## Practice: Extend the Q&A Agent

Now we combine multiple concepts together.

In this example:
- we save state
- load previous memory
- store user questions
- update conversation history

This creates a more realistic AI application.

```
Load Previous State
        ↓
Add User Question
        ↓
Run LLM Node
        ↓
Save Updated State
        ↓
Return Response
```

In [7]:
import json
import os
from langgraph.graph import StateGraph, END

STATE_FILE = "state.json"

# -----------------------------
# Persistence functions
# -----------------------------
def save_state(state):
    with open(STATE_FILE, "w") as f:
        json.dump(state, f)

    print("State saved")


def load_state():
    if os.path.exists(STATE_FILE):
        with open(STATE_FILE, "r") as f:
            return json.load(f)

    return {
        "messages": [],
        "memory": {}
    }

# Node
# -----------------------------
def extended_llm_node(state):

    state.setdefault("messages", [])
    state.setdefault("memory", {})

    user_message = state["messages"][-1]

    # Store memory
    state["memory"]["last_question"] = user_message

    response = f"Answer to: {user_message}"

    state["messages"].append(response)

    save_state(state)

    return state

# -----------------------------
# Graph
# -----------------------------
graph = StateGraph(dict)

graph.add_node("llm", extended_llm_node)

graph.set_entry_point("llm")

graph.add_edge("llm", END)

app = graph.compile()

# -----------------------------
# Load previous state
# -----------------------------
initial_state = load_state()

# Add user question
initial_state["messages"].append("What is LangGraph?")

# Run graph
result = app.invoke(initial_state)

print(result)    

State saved
{'messages': ['Hello', 'What is LangGraph?', 'Answer to: What is LangGraph?'], 'memory': {'key': 'value', 'last_question': 'What is LangGraph?'}, 'counter': 1}


These intermediate concepts are what make LangGraph suitable for real AI applications.

Because real-world AI systems need:
- memory
- error handling
- persistence
- multiple agents
- human interaction